In [ ]:
#| eval: false
! [ -e /content ] && pip install -Uqq xcube # upgrade xcube on colab

In [ ]:
#| default_exp l2r.data.info_gain

In [ ]:
#| export
from fastcore.basics import *
from fastai.torch_core import *
from fastai.data.core import *
from fastai.data.transforms import *
from fastai.text.core import *
from fastai.text.data import *
from xcube.imports import *
from xcube.torch_imports import *
from datasets import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

In [ ]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

# Information Gain

> Computation of mutual information gain

This module contains the all classes and functions needed to compute mutual information gain for the tokens and labels. This mutual information is then used to bootstrap a L2R model from xml text data. Please follow the [Boot L2R](14_tutorial.boot_l2r.ipynb) to understand how this module is used.

In [ ]:
#| export
class BatchLbsChunkify(ItemTransform):
    order = 100
    def __init__(self, chnk_st, chnk_end): store_attr('chnk_st,chnk_end')
    def encodes(self, x): 
        return (x[0], x[1][:, self.chnk_st:self.chnk_end])

In [ ]:
#| export
class OneHotInputTransformDL(DataLoader):
    """
    A DataLoader subclass that transforms input_ids into one-hot encodings.
    
    Args:
        dl: Original DataLoader from create_chunked_dataloaders (to copy attributes)
        tokenizer: Hugging Face tokenizer for vocab size
    """
    def __init__(self, dl, tokenizer, shuffle=False):
        # Copy attributes from the original DataLoader
        super().__init__(dl.dataset, batch_size=dl.batch_size, shuffle=shuffle, 
                         collate_fn=dl.collate_fn, drop_last=dl.drop_last)
        self.tokenizer = tokenizer
        self.vocab_size = len(tokenizer)
    
    def __iter__(self):
        # Get the base iterator from the parent DataLoader
        base_iterator = super().__iter__()
        
        # Apply transformation to each batch
        for batch in base_iterator:
            input_ids = batch['input_ids']  # Shape: [batch_size, max_length]
            one_hot_labels = batch['one_hot_labels']  # Shape: [batch_size, chunk_size]
            
            # # Create one-hot encoding for input_ids
            # # Shape: [batch_size, max_length, vocab_size]
            # one_hot_input_ids = torch.zeros(
            #     input_ids.shape[0], input_ids.shape[1], self.vocab_size, 
            #     device=input_ids.device
            # )
            # one_hot_input_ids.scatter_(2, input_ids.unsqueeze(-1), 1)
            
            # # Collapse to presence/absence: [batch_size, vocab_size]
            # one_hot_input_presence = one_hot_input_ids.max(dim=1)[0]
            
            # # Yield transformed batch: x = one_hot_input_presence, y = one_hot_labels
            # yield one_hot_input_presence, one_hot_labels

            # Initialize presence/absence tensor: [batch_size, vocab_size]
            one_hot_input_presence = torch.zeros(
                input_ids.shape[0], self.vocab_size, device=input_ids.device
            )
            
            # Set 1s for tokens present in input_ids
            for b in range(input_ids.shape[0]):  # Loop over batch
                # Get unique token IDs in this sequence
                unique_tokens = torch.unique(input_ids[b])
                # Set 1s for these tokens
                one_hot_input_presence[b, unique_tokens] = 1
            
            yield one_hot_input_presence, one_hot_labels


    def get_sample_batch(self):
        """
        Returns a sample batch without consuming the main iterator.
        
        Returns:
            Tuple (x, y): Sample batch with one-hot encoded input_ids and labels
        """
        # Create a temporary iterator to fetch one batch
        temp_iterator = super().__iter__()
        try:
            batch = next(temp_iterator)
            input_ids = batch['input_ids']
            one_hot_labels = batch['one_hot_labels']
            one_hot_input_ids = torch.zeros(
                input_ids.shape[0], input_ids.shape[1], self.vocab_size, 
                device=input_ids.device
            )
            one_hot_input_ids.scatter_(2, input_ids.unsqueeze(-1), 1)
            one_hot_input_presence = one_hot_input_ids.max(dim=1)[0]
            return one_hot_input_presence, one_hot_labels
        except StopIteration:
            raise ValueError("DataLoader is empty")

In [ ]:
#| export
def chunk_text(text: str, tokenizer: AutoTokenizer, max_length: int = 512, overlap: int = 50):
    """
    Split text into chunks that fit within max_length tokens, with optional overlap.

    Args:
        text (str): Input text to chunk.
        tokenizer (AutoTokenizer): Tokenizer to encode text.
        max_length (int): Maximum tokens per chunk (including special tokens).
        overlap (int): Number of tokens to overlap between chunks.

    Returns:
        List[Dict]: List of tokenized chunks, each with input_ids, attention_mask, etc.
    """
    # Tokenize the entire text without truncation
    encoded = tokenizer(
        text,
        add_special_tokens=False,  # Add [CLS], [SEP] per chunk later
        return_tensors="pt",
        return_attention_mask=True
    )
    
    input_ids = encoded["input_ids"][0]
    attention_mask = encoded["attention_mask"][0]
    
    # Account for [CLS] and [SEP] in max_length
    chunk_size = max_length - 2  # Reserve 1 for [CLS], 1 for [SEP]
    if chunk_size <= 0:
        raise ValueError("max_length must be at least 3 to accommodate special tokens")
    
    chunks = []
    i = 0
    while i < len(input_ids):
        # Determine chunk end, ensuring it doesn't exceed input length
        end = min(i + chunk_size, len(input_ids))
        
        # Extract chunk
        chunk_ids = input_ids[i:end]
        chunk_mask = attention_mask[i:end]
        
        # Add special tokens
        cls_id = tokenizer.cls_token_id or tokenizer.bos_token_id
        sep_id = tokenizer.sep_token_id or tokenizer.eos_token_id
        chunk_ids = torch.cat([
            torch.tensor([cls_id], dtype=chunk_ids.dtype),
            chunk_ids,
            torch.tensor([sep_id], dtype=chunk_ids.dtype)
        ])
        chunk_mask = torch.cat([
            torch.tensor([1], dtype=chunk_mask.dtype),
            chunk_mask,
            torch.tensor([1], dtype=chunk_mask.dtype)
        ])
        
        # Pad to max_length if needed
        padding_length = max_length - len(chunk_ids)
        if padding_length > 0:
            chunk_ids = torch.cat([
                chunk_ids,
                torch.zeros(padding_length, dtype=chunk_ids.dtype)
            ])
            chunk_mask = torch.cat([
                chunk_mask,
                torch.zeros(padding_length, dtype=chunk_mask.dtype)
            ])
        
        # Store chunk
        chunks.append({
            "input_ids": chunk_ids,
            "attention_mask": chunk_mask,
            "token_type_ids": torch.zeros_like(chunk_ids) if "token_type_ids" in tokenizer.model_input_names else None
        })
        
        # Move to next chunk with overlap
        i += chunk_size - overlap
    
    return chunks


class FlatChunkDataset(Dataset):
    """
    A PyTorch Dataset that treats each chunk as an individual item, with processed labels and one-hot encodings.
    """
    def __init__(self, dataset: Dataset, label_delim: Optional[str] = None):
        self.items = []
        for idx, item in enumerate(dataset):
            chunks = item["chunks"]
            if "labels" in item:
                label = item["labels"]
                if label_delim and isinstance(label, str):
                    label_list = label.split(label_delim)
                else:
                    label_list = [label] if isinstance(label, str) else label
            else:
                label_list = None
                logging.info(f"Data point {idx}: No labels found, setting label_list to None.")
            
            one_hot_labels = item.get("one_hot_labels", None)
            if one_hot_labels is None:
                logging.info(f"Data point {idx}: No one_hot_labels found, setting to None.")

            for chunk in chunks:
                chunk_item = {
                    "input_ids": chunk["input_ids"],
                    "attention_mask": chunk["attention_mask"],
                }
                if "token_type_ids" in chunk and chunk["token_type_ids"] is not None:
                    chunk_item["token_type_ids"] = chunk["token_type_ids"]
                if "labels" in item: chunk_item["labels"] = item["labels"]
                chunk_item["label_list"] = label_list
                chunk_item["one_hot_labels"] = one_hot_labels
                if "text" in item: chunk_item["text"] = item["text"]
                self.items.append(chunk_item)
        logging.info(f"Created FlatChunkDataset with {len(self.items)} chunks.")

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int | List[int]) -> Dict[str, Any] | List[Dict[str, Any]]:
        """
        Retrieve item(s) by index, supporting both integer and list of indices.
        """
        if isinstance(idx, (list, np.ndarray)):
            return [self.items[i] for i in idx]
        return self.items[idx]

    def __getitems__(self, indices):
        """Handle batched indices from DataLoader"""
        if isinstance(indices, dict):
            # Handle the case where indices is a dictionary
            # This depends on what format indices actually has
            return [self.items[i] for i in range(len(self.items)) if i in indices.values()]
        elif isinstance(indices, (list, np.ndarray)):
            return [self.items[i] for i in indices]
        else:
            return self.items[indices]

    def __repr__(self) -> str:
        num_items = len(self.items)
        keys = self.items[0].keys() if self.items else []
        return f"FlatChunkDataset(num_chunks={num_items}, keys={list(keys)})"

    
def check_chunk_labels(tokenized_dataset: Dataset, flat_dataset: 'FlatChunkDataset', label_delim: Optional[str] = None) -> bool:
    """
    Sanity check to ensure all chunks of a data point in FlatChunkDataset have the same
    'labels', 'label_list', and 'one_hot_labels' as the original data point.

    Uses the global 'logging' for outputting check results.

    Args:
        tokenized_dataset: Hugging Face Dataset with 'text', 'chunks', 'labels', 'label_list', 'one_hot_labels'.
        flat_dataset: FlatChunkDataset with chunks, 'labels', 'label_list', 'one_hot_labels'.
        label_delim: Optional delimiter used to process labels into label_list.

    Returns:
        bool: True if labels, label_list, and one_hot_labels are consistent for all chunks, False otherwise.
    """
    if "labels" not in tokenized_dataset.column_names:
        logging.info("No 'labels' column in tokenized_dataset. Skipping check.")
        return True

    chunk_counts = [len(item["chunks"]) for item in tokenized_dataset]
    chunk_start = 0
    labels_passed = True
    label_list_passed = True
    one_hot_passed = True

    for idx, (item, chunk_count) in enumerate(zip(tokenized_dataset, chunk_counts)):
        # Expected values from tokenized_dataset
        expected_labels = item["labels"]
        expected_label_list = item.get("label_list", None)
        expected_one_hot = item.get("one_hot_labels", None)
        
        # Process expected label_list for comparison
        if expected_labels is not None:
            if label_delim and isinstance(expected_labels, str):
                computed_label_list = expected_labels.split(label_delim)
            else:
                computed_label_list = [expected_labels] if isinstance(expected_labels, str) else expected_labels
        else:
            computed_label_list = None

        # Get chunk values from flat_dataset
        chunk_labels = [
            flat_dataset[chunk_start + i]["labels"]
            for i in range(chunk_count)
            if "labels" in flat_dataset[chunk_start + i]
        ]
        chunk_label_lists = [
            flat_dataset[chunk_start + i]["label_list"]
            for i in range(chunk_count)
            if "label_list" in flat_dataset[chunk_start + i]
        ]
        chunk_one_hots = [
            flat_dataset[chunk_start + i]["one_hot_labels"]
            for i in range(chunk_count)
            if "one_hot_labels" in flat_dataset[chunk_start + i]
        ]

        if not (chunk_labels or chunk_label_lists or chunk_one_hots):
            logging.info(f"Data point {idx}: No labels, label_list, or one_hot_labels found in chunks. Skipping.")
            chunk_start += chunk_count
            continue

        # Check labels
        for i, chunk_label in enumerate(chunk_labels):
            if isinstance(expected_labels, torch.Tensor) and isinstance(chunk_label, torch.Tensor):
                labels_equal = torch.equal(expected_labels, chunk_label)
            elif isinstance(expected_labels, list) and isinstance(chunk_label, list):
                labels_equal = expected_labels == chunk_label
            else:
                labels_equal = expected_labels == chunk_label
            if not labels_equal:
                logging.info(f"Data point {idx}, chunk {i + 1}: Labels mismatch. Expected {expected_labels}, got {chunk_label}")
                labels_passed = False

        # Check label_list
        for i, chunk_label_list in enumerate(chunk_label_lists):
            if expected_label_list is None and chunk_label_list is None:
                continue
            if expected_label_list is None or chunk_label_list is None:
                logging.info(f"Data point {idx}, chunk {i + 1}: Label_list mismatch. Expected {expected_label_list}, got {chunk_label_list}")
                label_list_passed = False
                continue
            if isinstance(expected_label_list, list) and isinstance(chunk_label_list, list):
                label_list_equal = expected_label_list == chunk_label_list
            else:
                label_list_equal = expected_label_list == chunk_label_list
            if not label_list_equal:
                logging.info(f"Data point {idx}, chunk {i + 1}: Label_list mismatch. Expected {expected_label_list}, got {chunk_label_list}")
                label_list_passed = False

        # Check one_hot_labels
        for i, chunk_one_hot in enumerate(chunk_one_hots):
            if expected_one_hot is None and chunk_one_hot is None:
                continue
            if expected_one_hot is None or chunk_one_hot is None:
                logging.info(f"Data point {idx}, chunk {i + 1}: One_hot_labels mismatch. Expected {expected_one_hot}, got {chunk_one_hot}")
                one_hot_passed = False
                continue
            # Convert to lists for comparison if necessary
            expected_one_hot_list = expected_one_hot if isinstance(expected_one_hot, list) else expected_one_hot.tolist()
            chunk_one_hot_list = chunk_one_hot if isinstance(chunk_one_hot, list) else chunk_one_hot.tolist()
            one_hot_equal = expected_one_hot_list == chunk_one_hot_list
            if not one_hot_equal:
                logging.info(f"Data point {idx}, chunk {i + 1}: One_hot_labels mismatch. Expected {expected_one_hot_list}, got {chunk_one_hot_list}")
                one_hot_passed = False

        chunk_start += chunk_count

    # Log results
    if labels_passed:
        logging.info(f"Labels check passed: All {chunk_start} chunks have consistent labels.")
    else:
        logging.info("Labels check failed: Some chunks have inconsistent labels.")
    if label_list_passed:
        logging.info(f"Label_list check passed: All {chunk_start} chunks have consistent label_list.")
    else:
        logging.info("Label_list check failed: Some chunks have inconsistent label_list.")
    if one_hot_passed:
        logging.info(f"One_hot_labels check passed: All {chunk_start} chunks have consistent one_hot_labels.")
    else:
        logging.info("One_hot_labels check failed: Some chunks have inconsistent one_hot_labels.")

    all_passed = labels_passed and label_list_passed and one_hot_passed
    logging.info(f"Overall check result: {'Passed' if all_passed else 'Failed'}")
    return all_passed


In [ ]:
#| export
from tqdm import tqdm
class MutualInfoGain:
    def __init__(self, df, tokenizer=None, mlb_classes=None, bs=8, chnk_sz=200, device=None, lbs_desc=None): 
        store_attr(but='lbs_desc')
        if lbs_desc is not None:
            try:
                with open(lbs_desc, 'rb') as f: self.lbs_desc = pickle.load(f)
            except FileNotFoundError as e: print(e)
    
    def onehotify_old(self, label_delim=None):
        x_tfms = [Tokenizer.from_df('text', n_workers=num_cpus()), attrgetter("text"), Numericalize(), OneHotEncode()]
        y_tfms = [ColReader('labels', label_delim = label_delim if label_delim is not None else ';'), MultiCategorize(), OneHotEncode()]
        tfms = [x_tfms, y_tfms]
        self.dsets = Datasets(self.df, tfms=[x_tfms, y_tfms], )
        self.toksize, self.lblsize = self.dsets.vocab.map(len)
        return self.dsets

    def onehotify(self, max_length=11776, label_delim=None, overlap=10):
        """
        Convert text and labels in a DataFrame to one-hot encoded format using Hugging Face datasets.
        
        Args:
            df: pandas DataFrame with 'text' and 'labels' columns
            tokenizer: Hugging Face tokenizer instance (e.g., from AutoTokenizer)
            max_length: maximum sequence length for tokenization (default: 11776)
            label_delim: delimiter for splitting labels (default: None, treats labels as single values)
        
        Returns:
            Dataset object with tokenized features and one-hot encoded labels
        """
        # Convert pandas DataFrame to Hugging Face Dataset
        dataset = Dataset.from_pandas(self.df)
        logging.info(f"The HuggingFace dataset created from pandas dataframe is of size {len(dataset)}")
        
        # Process labels
        def process_labels(examples):
            if label_delim:
                labels = [example.split(label_delim) for example in examples['labels']]
            else:
                labels = [[label] for label in examples['labels']]
            return {'label_list': labels}
        
        # Apply label processing
        dataset = dataset.map(process_labels, batched=True, desc="Processing Labels")
        
        # Initialize and fit MultiLabelBinarizer for one-hot encoding
        mlb = MultiLabelBinarizer(classes=self.mlb_classes)
        one_hot_labels = mlb.fit_transform(dataset['label_list'])
        
        # Add one-hot encoded labels efficiently using map
        def add_one_hot(examples, indices):
            return {'one_hot_labels': [one_hot_labels[idx].tolist() for idx in indices]}
        
        dataset = dataset.map(
            add_one_hot,
            with_indices=True,
            batched=True,
            desc="Adding one-hot labels"
        )
        
        # Tokenize text using the provided Hugging Face tokenizer
        def tokenize_function(examples):
            encodings = self.tokenizer(
                examples["text"],
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt"
            )
            return {
                'input_ids': encodings['input_ids'].tolist(),
                'attention_mask': encodings['attention_mask'].tolist(),
                'one_hot_labels': examples['one_hot_labels']
            }
        
        # Tokenize text using the provided Hugging Face tokenizer
        def tokenize_function_old(examples):
            all_chunks = []
            for text in examples["text"]:
                chunks = chunk_text(text, self.tokenizer, max_length, overlap)
                all_chunks.append(chunks)
            return {"chunks": all_chunks}
        
        # Apply tokenization
        dataset = dataset.map(tokenize_function, batched=True, desc="Tokenizing Texts")

        # Create FlatChunkDataset
        # flat_dataset = FlatChunkDataset(dataset, label_delim=label_delim)
        # logging.info(f"FlatChunkDataset: {flat_dataset}")
        # logging.info(f"FlatChunkDataset type: {type(flat_dataset)}")

        # Check labels consistency
        # logging.info("Running label consistency check...")
        # passed = check_chunk_labels(dataset, flat_dataset, label_delim=label_delim)
        # logging.info(f"Label check result: {'Passed' if passed else 'Failed'}")
        
        # Store sizes
        self.toksize = len(self.tokenizer)
        self.lblsize = len(mlb.classes_)
        
        # Clean up unnecessary columns
        # dataset = dataset.remove_columns(['label_list', 'text'])
        
        self.dsets = dataset        

        return self.dsets
        
    def lbs_chunked(self):
        import pdb; pdb.set_trace()
        lbs = self.dsets.vocab[1]
        self.dls = []
        for chnk_st in range(0, len(lbs), self.chnk_sz):
            self.dls.append(TfmdDL(self.dsets, bs=self.bs, 
                              after_batch=[BatchLbsChunkify(chnk_st, min(chnk_st+self.chnk_sz, len(lbs)))], 
                              device=default_device() if self.device is None else self.device))
        return self.dls

    def create_chunked_dataloaders(self, device=None):
        """
        Create multiple DataLoaders, each with a chunk of the one-hot encoded labels.
        
        Args:
            device: Device to load data onto (default: None, uses default device)
        
        Returns:
            List of DataLoaders, each handling a chunk of labels
        """
        # Set default device if none provided
        device = device if device is not None else default_device()
        
        # Calculate number of chunks
        num_labels = len(self.mlb_classes)
        num_chunks = (num_labels + self.chnk_sz - 1) // self.chnk_sz  # Ceiling division
        logging.info(f"Chunking the labels into {num_chunks} many chunks, each chunk has up to {self.chnk_sz} many labels.")
        
        # Custom collate function for chunking labels
        def chunk_collate_fn(chunk_start, chunk_end):
            def collate_fn_old(batch):
                # Extract input_ids, attention_mask, and one_hot_labels from batch
                input_ids = torch.tensor([item['input_ids'] for item in batch], dtype=torch.long)
                attention_mask = torch.tensor([item['attention_mask'] for item in batch], dtype=torch.long)
                one_hot_labels = torch.tensor([item['one_hot_labels'] for item in batch], dtype=torch.float32)
                
                # Chunk the one-hot labels
                chunked_labels = one_hot_labels[:, chunk_start:chunk_end]
                
                # Move to device
                return {
                    'input_ids': input_ids.to(device),
                    'attention_mask': attention_mask.to(device),
                    'one_hot_labels': chunked_labels.to(device)
                }

            
            def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
                """
                Simplified collate function that directly creates tensors from batch items.
                """
                # Create batched tensors for inputs that should always be present
                batched = {
                    "input_ids": torch.tensor([item["input_ids"] for item in batch], dtype=torch.long, device=device),
                    "attention_mask": torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long, device=device),
                }
                
                # Add token_type_ids if present
                if all("token_type_ids" in item for item in batch):
                    batched["token_type_ids"] = torch.tensor([item["token_type_ids"] for item in batch], dtype=torch.long, device=device)
                else:
                    pass
                    # logging.warning("Some items in batch are missing token_type_ids")
                
                # Add labels if one_hot_labels are present
                if all("one_hot_labels" in item for item in batch):
                    batched["one_hot_labels"] = torch.tensor([item["one_hot_labels"] for item in batch], dtype=torch.float, device=device)
                else:
                    # Fallback for missing one_hot_labels
                    batched["one_hot_labels"] = torch.zeros((len(batch), 1), dtype=torch.float, device=device)
                    logging.warning(f"Batch has no valid one_hot_labels, using zero tensor.")

                # Chunk the one-hot labels
                batched["one_hot_labels"] = batched["one_hot_labels"][:, chunk_start:chunk_end]
                
                return batched

            return collate_fn
        
        # Create list of DataLoaders
        self.dataloaders = []
        for chunk_start in range(0, num_labels, self.chnk_sz):
            chunk_end = min(chunk_start + self.chnk_sz, num_labels)
            dl = DataLoader(
                self.dsets,
                batch_size=self.bs,
                shuffle=False,  # Optional: set to False if you don't want shuffling
                collate_fn=chunk_collate_fn(chunk_start, chunk_end)
            )
            self.dataloaders.append(dl)
        
        return self.dataloaders

    def sanity_check_chunked(self):
        """
        Perform sanity checks on the chunked DataLoaders using logging.
        
        Args:
            dataset: Processed Dataset object from onehotify
            mlb_classes: List of predefined label classes
            chunk_size: Number of labels per chunk (default: 5)
            batch_size: Batch size for each DataLoader (default: 2)
        
        Returns:
            None (logs results of sanity checks)
        """
        # Create chunked DataLoaders
        dls = self.create_chunked_dataloaders()
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        logging.info("Running sanity checks for chunked DataLoaders...")
        
        # Check 1: Verify the first DataLoader is a PyTorch DataLoader
        assert isinstance(dls[0], DataLoader), "First DataLoader should be a torch.utils.data.DataLoader"
        logging.info("Check 1: DataLoader type - PASSED")
        
        # Check 2: Verify the number of DataLoaders matches the expected number of chunks
        expected_num_chunks = np.ceil(len(self.mlb_classes) / self.chnk_sz)
        assert len(dls) == expected_num_chunks, f"Expected {expected_num_chunks} DataLoaders, got {len(dls)}"
        logging.info(f"Check 2: Number of DataLoaders ({len(dls)} == {expected_num_chunks}) - PASSED")
        
        # Check 3: Verify the length of the first DataLoader (number of batches)
        expected_batches = np.ceil(len(self.dsets) / self.bs)  # drop_last=False by default
        assert len(dls[0]) == expected_batches, f"Expected {expected_batches} batches, got {len(dls[0])}"
        logging.info(f"Check 3: Number of batches in first DataLoader ({len(dls[0])} == {expected_batches}) - PASSED")
        
        # Check 4: Verify that labels are correctly split and can be reconstructed
        # Get the first item from the dataset for comparison
        sample = self.dsets[0]
        original_labels = torch.tensor(sample['one_hot_labels'], dtype=torch.float32).to(device)
        
        # Collect the first item's labels from all DataLoaders
        chunked_labels = []
        for dl in dls:
            for batch in itertools.islice(dl, 1):  # Take only the first batch
                chunked_labels.append(batch['one_hot_labels'][0])  # First item in batch
                break
        
        # Concatenate the chunked labels
        reconstructed_labels = torch.cat(chunked_labels)
        
        # Compare with original
        assert torch.allclose(reconstructed_labels, original_labels), "Reconstructed labels do not match original"
        logging.info("Check 4: Label reconstruction across chunks - PASSED")
        logging.info(f"Original labels shape: {original_labels.shape}")
        logging.info(f"Reconstructed labels shape: {reconstructed_labels.shape}")

        # Check 5: Verify one-hot encoding of input_ids based on tokenizer vocab
        batch = next(iter(dls[0]))  # Get first batch from first DataLoader
        input_ids = batch['input_ids']  # Shape: [batch_size, max_length]
        vocab_size = len(self.tokenizer)
        
        # Create one-hot encoding: [batch_size, max_length, vocab_size]
        one_hot_input_ids = torch.zeros(input_ids.shape[0], input_ids.shape[1], vocab_size, device=input_ids.device)
        one_hot_input_ids.scatter_(2, input_ids.unsqueeze(-1), 1)
        
        # Collapse to presence/absence: [batch_size, vocab_size]
        one_hot_presence = one_hot_input_ids.max(dim=1)[0]
        
        # Test for first item in batch
        item_idx = 0
        present_token_ids = torch.where(one_hot_presence[item_idx] == 1)[0]
        unique_input_ids = torch.unique(input_ids[item_idx])  # Unique tokens in sequence
        assert torch.all(one_hot_presence[item_idx][unique_input_ids] == 1), "One-hot encoding misses present tokens"
        logging.info("Check 5: One-hot encoding of input_ids - PASSED")
        logging.info(f"Vocab size: {vocab_size}")
        logging.info(f"Number of unique tokens present in first item: {len(present_token_ids)}")
        
        logging.info("All sanity checks passed!")

    def _mutual_info_gain_old(self, dl, parent_bar=None):
        """
        Computes [mutual information gain](https://en.wikipedia.org/wiki/Mutual_information) for each token label pair
        `dl` is (bag-of-words text, one-hot encoded targets)
        """
        xb, yb = dl.one_batch() 
        toksize, lblsize = xb.size(1), yb.size(1)
        p_TL = torch.zeros(toksize, lblsize, 4, dtype=torch.float, device=default_device())
        eps = p_TL.new_empty(1).fill_(1e-8)
        pbar = progress_bar(dl, parent=parent_bar, leave=True)
        for x,y in pbar:
            test_eq(x.shape, (dl.bs, toksize)); test_eq(y.shape, (dl.bs, lblsize))
            t = x.unsqueeze(-1).expand(-1, -1, lblsize) ; test_eq(t.shape, (dl.bs, toksize, lblsize))
            l = y.unsqueeze(1).expand(-1, toksize, -1) ; test_eq(l.shape, (dl.bs, toksize, lblsize))
            tl = torch.stack((t,l), dim=-1) ; test_eq(tl.shape, (dl.bs, toksize, lblsize, 2)) 
            p_TL_tt = tl[...,0].logical_and(tl[...,1]) ; test_eq(p_TL_tt.shape, (dl.bs, toksize, lblsize)) 
            p_TL_tf = tl[...,0].logical_and(tl[...,1].logical_not()) ; test_eq(p_TL_tf.shape, (dl.bs, toksize, lblsize)) 
            p_TL_ft = tl[...,0].logical_not().logical_and(tl[...,1]) ; test_eq(p_TL_ft.shape, (dl.bs, toksize, lblsize))
            p_TL_ff = tl[...,0].logical_not().logical_and(tl[...,1].logical_not()) ; test_eq(p_TL_ff.shape, (dl.bs, toksize, lblsize)) 
            p_TL = p_TL + torch.stack((p_TL_tt, p_TL_tf, p_TL_ft, p_TL_ff), dim=-1).float().sum(dim=0)
        p_TL = p_TL / tensor(len(self.dsets)).float()
        p_TL = p_TL.view(toksize, lblsize, 2, 2) ; test_eq(p_TL.shape, (toksize, lblsize, 2, 2))# last axis: lbl axis, 2nd last axis: token axis
        return p_TL

    def _mutual_info_gain(self, dl, parent_bar=None):
        """
        Computes mutual information gain for each token-label pair.
        `dl` is a DataLoader from create_chunked_dataloaders, transformed to yield
        (one-hot encoded input_ids presence, one-hot encoded targets).
        
        Args:
            dl: Original DataLoader from create_chunked_dataloaders
            tokenizer: Hugging Face tokenizer for vocab size
            parent_bar: Optional parent progress bar (not used here, kept for compatibility)
        
        Returns:
            p_TL: Tensor of shape [toksize, lblsize, 2, 2] with joint probabilities
        """
        # Transform the DataLoader
        transformed_dl = OneHotInputTransformDL(dl, self.tokenizer)
        
        # Get a sample batch to determine sizes without consuming the main iterator
        xb, yb = transformed_dl.get_sample_batch()
        toksize, lblsize = xb.size(1), yb.size(1)  # toksize = vocab_size, lblsize = chunk_sizesizes without consuming a batch

        # Initialize joint probability tensor
        p_TL = torch.zeros(toksize, lblsize, 4, dtype=torch.float, device=xb.device)
        
        # Iterate over the transformed DataLoader
        pbar = progress_bar(transformed_dl, parent=parent_bar, leave=True)
        for x, y in pbar:
            batch_size = x.size(0)  # Actual batch size, may be <= transformed_dl.batch_size
            assert x.shape[0] <= transformed_dl.batch_size and x.shape[1] == toksize, f"Expected x shape[0] <= {transformed_dl.batch_size} and shape[1] = {toksize}, got {x.shape}"
            assert y.shape[0] <= transformed_dl.batch_size and y.shape[1] == lblsize, f"Expected y shape[0] <= {transformed_dl.batch_size} and shape[1] = {lblsize}, got {y.shape}"

            # Expand dimensions for broadcasting
            t = x.unsqueeze(-1).expand(-1, -1, lblsize)  # [bs, toksize, lblsize]
            assert t.shape == (batch_size, toksize, lblsize), f"Expected t shape {(batch_size, toksize, lblsize)}, got {t.shape}"
            
            l = y.unsqueeze(1).expand(-1, toksize, -1)  # [bs, toksize, lblsize]
            assert l.shape == (batch_size, toksize, lblsize), f"Expected l shape {(batch_size, toksize, lblsize)}, got {l.shape}"
            
            # Stack token and label presence
            tl = torch.stack((t, l), dim=-1)  # [bs, toksize, lblsize, 2]
            assert tl.shape == (batch_size, toksize, lblsize, 2), f"Expected tl shape {(batch_size, toksize, lblsize, 2)}, got {tl.shape}"
            
            # Compute joint probabilities
            p_TL_tt = tl[..., 0].logical_and(tl[..., 1])  # [bs, toksize, lblsize]
            assert p_TL_tt.shape == (batch_size, toksize, lblsize), f"Expected p_TL_tt shape {(batch_size, toksize, lblsize)}, got {p_TL_tt.shape}"
            
            p_TL_tf = tl[..., 0].logical_and(tl[..., 1].logical_not())  # [bs, toksize, lblsize]
            assert p_TL_tf.shape == (batch_size, toksize, lblsize), f"Expected p_TL_tf shape {(batch_size, toksize, lblsize)}, got {p_TL_tf.shape}"
            
            p_TL_ft = tl[..., 0].logical_not().logical_and(tl[..., 1])  # [bs, toksize, lblsize]
            assert p_TL_ft.shape == (batch_size, toksize, lblsize), f"Expected p_TL_ft shape {(batch_size, toksize, lblsize)}, got {p_TL_ft.shape}"
            
            p_TL_ff = tl[..., 0].logical_not().logical_and(tl[..., 1].logical_not())  # [bs, toksize, lblsize]
            assert p_TL_ff.shape == (batch_size, toksize, lblsize), f"Expected p_TL_ff shape {(batch_size, toksize, lblsize)}, got {p_TL_ff.shape}"
            
            # Accumulate counts
            p_TL = p_TL + torch.stack((p_TL_tt, p_TL_tf, p_TL_ft, p_TL_ff), dim=-1).float().sum(dim=0)
        
        # Normalize by dataset size
        p_TL = p_TL / torch.tensor(len(self.dsets), dtype=torch.float, device=p_TL.device)
        p_TL = p_TL.view(toksize, lblsize, 2, 2)  # Reshape to [toksize, lblsize, 2, 2]
        assert p_TL.shape == (toksize, lblsize, 2, 2), f"Expected p_TL shape {(toksize, lblsize, 2, 2)}, got {p_TL.shape}"
        
        return p_TL

    def joint_pmf(self):
        self.p_TL_full = []
        mb = master_bar(self.dataloaders)
        for i,dl in enumerate(mb):
            p_TL = self._mutual_info_gain(dl,parent_bar=mb)
            self.p_TL_full.append(p_TL)
            del p_TL; #del p_T; del p_L; del p_TxL; del I_TL; torch.cuda.empty_cache()
        self.p_TL_full = torch.cat(self.p_TL_full, dim=1); 
        assert self.p_TL_full.shape == (self.toksize, self.lblsize, 2, 2), f"Expected p_TL_full shape {(self.toksize, self.lblsize, 2, 2)}, got {self.p_TL_full.shape}"
        return self.p_TL_full
    
    def compute(self):
        eps = self.p_TL_full.new_empty(1).fill_(1e-15)
        toksize, lblsize = self.p_TL_full.size(0), self.p_TL_full.size(1)
        p_T = self.p_TL_full[:,0].sum(-1, keepdim=True); test_eq(p_T.shape, (toksize, 2, 1))# 0 because we can pick any label and apply total prob law
        p_L = self.p_TL_full[0,:].sum(-2, keepdim=True); test_eq(p_L.shape, (lblsize, 1, 2)) # 0 becuase we can pick any token and apply total prob law
        p_TxL = self.p_TL_full.sum(-1, keepdim=True) @ self.p_TL_full.sum(-2, keepdim=True); test_eq(p_TxL.shape, (toksize, lblsize, 2, 2))
        H_T = -(p_T * torch.log(p_T+eps)).sum(-2).squeeze(); test_eq(H_T.shape, [toksize])
        H_L = -(p_L * torch.log(p_L+eps)).sum(-1).squeeze(); test_eq(H_L.shape, [lblsize])
        I_TL = (self.p_TL_full * torch.log((self.p_TL_full + eps)/(p_TxL + eps))).flatten(start_dim=-2).sum(-1); test_eq(I_TL.shape, (toksize, lblsize))
        return p_T, p_L, p_TxL, H_T, H_L, I_TL
    
    @property
    def lbs_frqs(self):
        f = ColReader('labels', label_delim=';')
        self._frqs = Counter()
        for o in self.df.itertuples(): self._frqs.update(f(o))
        return self._frqs

In [ ]:
@property
@patch
def lbs_frqs(self:MutualInfoGain):
    f = ColReader('labels', label_delim=';')
    self._frqs = Counter()
    for o in self.df.itertuples(): self._frqs.update(f(o))
    return self._frqs

In [ ]:
#| export
@patch
def _gen(self:MutualInfoGain, p_TL, p_T, p_L, info, H_T, H_L, k=5):
    toks, lbs = array(self.dsets.vocab[0]), self.dsets.vocab[1]
    sorted_by_tok, tok_idxs = torch.sort(info, dim=0, descending=True) 
    for i,o in enumerate(lbs):
        topk_tok_idxs = tok_idxs[:k, i].cpu()
        topk_toks = toks[topk_tok_idxs]
        topk_toks_probs = p_T.squeeze()[:,0][topk_tok_idxs].cpu().numpy()
        topk_info_gains = sorted_by_tok[:k, i].cpu().numpy()
        topk_jnt_probs = p_TL[topk_tok_idxs, [i]][:,0,0].cpu().numpy()
        lbl_entropy = H_L[i].cpu().numpy()
        topk_tok_entrops = H_T[topk_tok_idxs].cpu().numpy()
        yield (o, self.lbs_frqs[o], p_L[i][0,0].cpu().numpy(), lbl_entropy, self.lbs_desc.get(o, 'Not Found'), 
               array(list(zip(topk_toks, topk_toks_probs, topk_tok_entrops, topk_jnt_probs, topk_info_gains))))

In [ ]:
#| export
@patch
def show(self:MutualInfoGain, *args, save_as=None, **kwargs):
    _data = self._gen(*args, **kwargs)
    df = pd.DataFrame(_data, columns=['label', 'freq', 'prob', 'entropy', 'description', 'top-k (token, prob, entropy, joint, info)'],)
    df[['prob', 'entropy',]] = df[['prob', 'entropy']].astype(np.float)
    df[['top-k (token, prob, entropy, joint, info)']] = df[['top-k (token, prob, entropy, joint, info)']].astype(np.str_) 
    if save_as is not None: df.to_feather(save_as)
    return df

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()